# Information
This script looks at the barplot generated as result of feature classification (again performed on HPC). For reasons of visual clarity, the classifications were aggregated per timepoint (script agecollapse.sh, which uses a slightly modified metadata file containing different host_age lables) before generating the qzv file. 

# 0. Setup
load packages
specify paths

In [1]:
import qiime2 as q2
import pandas as pd

In [2]:
meta_dir="../../data"
data_dir="../../output/02_taxonomicclassification"
output_dir="../../output/02_taxonomicclassification/grouped"

# 1. summed by age (s. script agecollapse.sh)

In [3]:
q2.Visualization.load(f"{output_dir}/bracken-barplot-bytimepoint.qzv")

<visualization: Visualization uuid: 678a31c0-3b1a-41d4-b654-c2394a1ac60f>

# 2. summed by age, subset by delivery mode and diet

In [4]:
meta = pd.read_csv(f"{meta_dir}/metadata.tsv", sep="\t", index_col=0)  
meta.head()

,collection_date,country,instrument,sample_number,delivery_mode,host_sex,diet_milk,city,latitude,longitude,urca_code,urca_description,host_id,host_age,seq_depth,age_months
sample-id,,,,,,,,,,,,,,,,
SRR25052523,19.09.2018,usa,illumina novaseq 6000,1,csection,female,NaN,falls church,38.86,-77.23,9,<1 hour to large city (1 - 5 million),10,2,17325587,2.0
SRR25052512,20.12.2018,usa,illumina novaseq 6000,2,csection,female,mixed,falls church,38.86,-77.23,9,<1 hour to large city (1 - 5 million),10,6,12407450,6.0
SRR25052501,08.07.2019,usa,illumina novaseq 6000,3,csection,female,mixed,falls church,38.86,-77.23,9,<1 hour to large city (1 - 5 million),10,12,11964978,12.0
SRR25052458,30.06.2020,usa,illumina novaseq 6000,4,csection,female,mixed,falls church,38.86,-77.23,9,<1 hour to large city (1 - 5 million),10,24,10921526,24.0
SRR25052720,28.08.2018,usa,illumina novaseq 6000,5,vaginal,male,NaN,falls church,38.86,-77.23,9,<1 hour to large city (1 - 5 million),105,0,12988703,0.0


In [5]:
meta["group"] = meta["delivery_mode"] + "_" + meta["diet_milk"]
meta["age_group"] = meta["host_age"].astype(int).astype(str).str.zfill(2) + "__" + meta["group"]

ok = meta["age_group"].notna()
meta[ok].to_csv(f"{output_dir}/metadata_bydeliverydiet.tsv", sep="\t", index_label="sample-id")

ok = meta["age_group"].notna()
meta_ok = meta[ok]
meta_ok.to_csv(f"{output_dir}/metadata_by_deliverydiet.tsv", sep="\t", index_label="sample-id")
print(ok.sum(), "of", len(meta), "samples kept")

143 of 234 samples kept


In [6]:
! qiime feature-table filter-samples \
  --i-table $data_dir/feature-table.qza \
  --m-metadata-file $output_dir/metadata_by_deliverydiet.tsv \
  --o-filtered-table $output_dir/feature-table-nodietNA.qza

Saved FeatureTable[Frequency] to: ../../output/02_taxonomicclassification/grouped/feature-table-nodietNA.qza


In [7]:
! qiime feature-table group \
  --i-table $output_dir/feature-table-nodietNA.qza \
  --p-axis sample \
  --m-metadata-file $output_dir/metadata_by_deliverydiet.tsv \
  --m-metadata-column age_group \
  --p-mode sum \
  --o-grouped-table $output_dir/table-by-deliverydiet.qza

Saved FeatureTable[Frequency] to: ../../output/02_taxonomicclassification/grouped/table-by-deliverydiet.qza


In [10]:
t = q2.Artifact.load(f"{output_dir}/table-by-deliverydiet.qza").view(pd.DataFrame)
for g in ["vaginal_bd", "vaginal_mixed", "csection_bd", "csection_mixed"]:
    ids = [i for i in t.index if i.endswith(f"__{g}")]
    pd.DataFrame({"sample-id": ids}).to_csv(f"{output_dir_dir}/ids_{g}.tsv", sep="\t", index=False)

In [14]:
%%bash

output_dir="../../output/02_taxonomicclassification/grouped"
set -e

for g in vaginal_bd vaginal_mixed csection_bd csection_mixed; do
  qiime feature-table filter-samples \
    --i-table $output_dir/table-by-deliverydiet.qza \
    --m-metadata-file $output_dir/ids_${g}.tsv \
    --o-filtered-table $output_dir/table-${g}.qza

  qiime taxa barplot \
    --i-table $output_dir/table-${g}.qza \
    --i-taxonomy $output_dir/../taxonomy-table.qza \
    --o-visualization $output_dir/barplot-${g}.qzv
done

Saved FeatureTable[Frequency] to: ../../output/02_taxonomicclassification/grouped/table-vaginal_bd.qza
Saved Visualization to: ../../output/02_taxonomicclassification/grouped/barplot-vaginal_bd.qzv
Saved FeatureTable[Frequency] to: ../../output/02_taxonomicclassification/grouped/table-vaginal_mixed.qza
Saved Visualization to: ../../output/02_taxonomicclassification/grouped/barplot-vaginal_mixed.qzv
Saved FeatureTable[Frequency] to: ../../output/02_taxonomicclassification/grouped/table-csection_bd.qza
Saved Visualization to: ../../output/02_taxonomicclassification/grouped/barplot-csection_bd.qzv
Saved FeatureTable[Frequency] to: ../../output/02_taxonomicclassification/grouped/table-csection_mixed.qza
Saved Visualization to: ../../output/02_taxonomicclassification/grouped/barplot-csection_mixed.qzv
